## ライブラリのインポート

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import japanize_matplotlib
import sys
from pathlib import Path
from sklearn.pipeline import Pipeline
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

sys.path.append(str(Path.cwd().parent))

## 万博用のデータ作成

In [2]:
from pipeline.etl.merge_df import load_base_data
from pipeline.features.make_lag_features import make_lag_features
from pipeline.features.make_event_flag import make_event_flag
from pipeline.features.make_station_distance_features import make_station_distance_features

df2 = load_base_data()
df2 = make_lag_features(df2)
df2 = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df2)
df2 = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df2)
df2 = make_station_distance_features(df2)

/Users/shoki_u/ml-training/objective/mlit_property_price_prediction/.venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [3]:
# df2 のシェイプを確認
print(df2.shape)
# df2 のカラムの確認
print(df2.columns.tolist())
# 此花区の行を取り出し
df2[df2['municipality'] == "此花区"].head()

(43804, 46)
['id', 'type', 'price_info_class', 'municipality_code', 'municipality', 'nearest_station', 'layout', 'structure', 'use', 'future_use', 'city_planning', 'renovation', 'transaction_notes', 'transaction_price', 'coverage_ratio', 'floor_area_ratio', 'area_sqm', 'area_no_max_limit_flag', 'building_year', 'building_before_war_flag', 'time_range_write_flag', 'time_no_max_limit_flag', 'time_to_station', 'transaction_year', 'transaction_quarter', 'actual_foreign_guests', 'foreign_male_counts', 'foreign_female_counts', 'total_counts', 'include_foreign_household_counts', 'facility_count', 'price_per_sqm', 'median_price_1year_ago', 'median_price_2year_ago', 'median_price_3year_ago', 'median_price_4year_ago', 'median_price_5year_ago', 'expo_announcement_flag', 'expo_development_start_flag', 'expo_development_completion_flag', 'expo_implementation_flag', 'ir_announcement_flag', 'ir_development_start_flag', 'ir_development_completion_flag', 'ir_implementation_flag', 'distance_to_yumeshima

,id,type,price_info_class,municipality_code,municipality,nearest_station,layout,structure,use,future_use,...,median_price_5year_ago,expo_announcement_flag,expo_development_start_flag,expo_development_completion_flag,expo_implementation_flag,ir_announcement_flag,ir_development_start_flag,ir_development_completion_flag,ir_implementation_flag,distance_to_yumeshima_km
13,8dba9cfb-d79f-431e-84a3-9af963792a4d,中古マンション等,不動産取引価格情報,27104,此花区,安治川口,３ＬＤＫ,ＲＣ,住宅,住宅,...,342857.142857,1,1,0,0,1,0,0,0,5.553492
96,0e65f001-07d5-48b8-93c6-6d88ba1130da,中古マンション等,不動産取引価格情報,27104,此花区,ユニバーサルシティ,NaN,ＲＣ,NaN,住宅,...,271794.871795,1,1,0,0,1,0,0,0,4.833863
97,bdd35f75-3850-4711-80ee-df03f2d6c1ea,中古マンション等,不動産取引価格情報,27104,此花区,ユニバーサルシティ,NaN,ＳＲＣ,NaN,NaN,...,271794.871795,1,1,0,0,1,0,0,0,4.833863
98,f932d068-3099-4429-bedc-773ef7a8280d,中古マンション等,不動産取引価格情報,27104,此花区,西九条,３ＬＤＫ,ＲＣ,住宅,住宅,...,271794.871795,1,1,0,0,1,0,0,0,7.776143
166,ab00cfbc-13f3-4656-95e1-62352867ff93,中古マンション等,不動産取引価格情報,27104,此花区,伝法,３ＬＤＫ,ＳＲＣ,住宅,住宅,...,261538.461538,1,1,0,0,1,0,0,0,7.039255


In [4]:
print([c for c in df2.columns if 'completion' in c or 'announcement' in c or 'implementation' in c or 'start' in c])

['expo_announcement_flag', 'expo_development_start_flag', 'expo_development_completion_flag', 'expo_implementation_flag', 'ir_announcement_flag', 'ir_development_start_flag', 'ir_development_completion_flag', 'ir_implementation_flag']


## IR用の予測データ作成

In [5]:
from pipeline.features.make_df3 import make_df3
df3 = make_df3(df2)

# df3のシェイプ確認
print('【df3のshape】')
print(df3.shape)
print('----' * 10)

【df3のshape】
(43804, 44)
----------------------------------------


In [6]:
# expo_*_flagが全て1になっていることを確認
muni_list = ['此花区', '鶴見区', '住之江区', '淀川区', '福島区']
for muni in muni_list:
    print(f'【{muni}のデータ総数】')
    print(len(df3[df3['municipality'] == muni]))
    print(f'【{muni}のexpo_*_flagの数の確認】')
    print(df3[df3['municipality'] == muni][['expo_announcement_flag',
       'expo_development_start_flag', 'expo_development_completion_flag',
       'expo_implementation_flag']].value_counts())

【此花区のデータ総数】
536
【此花区のexpo_*_flagの数の確認】
expo_announcement_flag  expo_development_start_flag  expo_development_completion_flag  expo_implementation_flag
1                       1                            1                                 1                           536
Name: count, dtype: int64
【鶴見区のデータ総数】
805
【鶴見区のexpo_*_flagの数の確認】
expo_announcement_flag  expo_development_start_flag  expo_development_completion_flag  expo_implementation_flag
1                       0                            0                                 0                           805
Name: count, dtype: int64
【住之江区のデータ総数】
1141
【住之江区のexpo_*_flagの数の確認】
expo_announcement_flag  expo_development_start_flag  expo_development_completion_flag  expo_implementation_flag
0                       1                            1                                 0                           1141
Name: count, dtype: int64
【淀川区のデータ総数】
4287
【淀川区のexpo_*_flagの数の確認】
expo_announcement_flag  expo_development_start_flag  expo_developmen

In [7]:
# ir_*_flagの確認
df3.pivot_table(values=['ir_announcement_flag',
       'ir_development_start_flag', 'ir_development_completion_flag',
       'ir_implementation_flag'], index='municipality', columns='transaction_year', aggfunc='median')

ir_announcement_flag                      \
transaction_year                 2026 2027 2028 2029 2030   
municipality                                                
中央区                               0.0  0.0  0.0  0.0  0.0   
住之江区                              0.0  0.0  0.0  0.0  0.0   
住吉区                               0.0  0.0  0.0  0.0  0.0   
北区                                0.0  0.0  0.0  0.0  0.0   
城東区                               0.0  0.0  0.0  0.0  0.0   
大正区                               0.0  0.0  0.0  0.0  0.0   
天王寺区                              0.0  0.0  0.0  0.0  0.0   
平野区                               0.0  0.0  0.0  0.0  0.0   
旭区                                0.0  0.0  0.0  0.0  0.0   
東住吉区                              0.0  0.0  0.0  0.0  0.0   
東成区                               0.0  0.0  0.0  0.0  0.0   
東淀川区                              0.0  0.0  0.0  0.0  0.0   
此花区                               1.0  1.0  1.0  1.0  1.0   
浪速区                               0.0  0.0  0.0  0.0  0.0   
淀川区                               0.0  0.0  0.0  0.0  0.0   
港区                                0.0  0.0  0.0  0.0  0.0   
生野区                               0.0  0.0  0.0  0.0  0.0   
福島区                               0.0  0.0  0.0  0.0  0.0   
西区                                0.0  0.0  0.0  0.0  0.0   
西成区                               0.0  0.0  0.0  0.0  0.0   
西淀川区                              0.0  0.0  0.0  0.0  0.0   
都島区                               0.0  0.0  0.0  0.0  0.0   
阿倍野区                              0.0  0.0  0.0  0.0  0.0   
鶴見区                               0.0  0.0  0.0  0.0  0.0   

                 ir_development_completion_flag                      \
transaction_year                           2026 2027 2028 2029 2030   
municipality                                                          
中央区                                         0.0  0.0  0.0  0.0  0.0   
住之江区                                        0.0  0.0  0.0  0.0  0.0   
住吉区                                         0.0  0.0  0.0  0.0  0.0   
北区                                          0.0  0.0  0.0  0.0  0.0   
城東区                                         0.0  0.0  0.0  0.0  0.0   
大正区                                         0.0  0.0  0.0  0.0  0.0   
天王寺区                                        0.0  0.0  0.0  0.0  0.0   
平野区                                         0.0  0.0  0.0  0.0  0.0   
旭区                                          0.0  0.0  0.0  0.0  0.0   
東住吉区                                        0.0  0.0  0.0  0.0  0.0   
東成区                                         0.0  0.0  0.0  0.0  0.0   
東淀川区                                        0.0  0.0  0.0  0.0  0.0   
此花区                                         0.0  0.0  0.0  0.0  1.0   
浪速区                                         0.0  0.0  0.0  0.0  0.0   
淀川区                                         0.0  0.0  0.0  0.0  0.0   
港区                                          0.0  0.0  0.0  0.0  0.0   
生野区                                         0.0  0.0  0.0  0.0  0.0   
福島区                                         0.0  0.0  0.0  0.0  0.0   
西区                                          0.0  0.0  0.0  0.0  0.0   
西成区                                         0.0  0.0  0.0  0.0  0.0   
西淀川区                                        0.0  0.0  0.0  0.0  0.0   
都島区                                         0.0  0.0  0.0  0.0  0.0   
阿倍野区                                        0.0  0.0  0.0  0.0  0.0   
鶴見区                                         0.0  0.0  0.0  0.0  0.0   

                 ir_development_start_flag                      \
transaction_year                      2026 2027 2028 2029 2030   
municipality                                                     
中央区                                    0.0  0.0  0.0  0.0  0.0   
住之江区                                   0.0  0.0  0.0  0.0  0.0   
住吉区                                    0.0  0.0  0.0  0.0  0.0   
北区                                     0.0  0.0  0.0  0

## 予測の実装

### 前処理パイプラインの構築

In [8]:
# df2のカラムの確認
df2.nunique()

id                                  43804
type                                    1
price_info_class                        1
municipality_code                      24
municipality                           24
nearest_station                       212
layout                                 35
structure                               6
use                                    10
future_use                              4
city_planning                          12
renovation                              2
transaction_notes                       5
transaction_price                     222
coverage_ratio                          4
floor_area_ratio                       13
area_sqm                               63
area_no_max_limit_flag                  1
building_year                          66
building_before_war_flag                2
time_range_write_flag                   2
time_no_max_limit_flag                  1
time_to_station                        31
transaction_year                  

In [9]:
df2.columns

Index(['id', 'type', 'price_info_class', 'municipality_code', 'municipality',
       'nearest_station', 'layout', 'structure', 'use', 'future_use',
       'city_planning', 'renovation', 'transaction_notes', 'transaction_price',
       'coverage_ratio', 'floor_area_ratio', 'area_sqm',
       'area_no_max_limit_flag', 'building_year', 'building_before_war_flag',
       'time_range_write_flag', 'time_no_max_limit_flag', 'time_to_station',
       'transaction_year', 'transaction_quarter', 'actual_foreign_guests',
       'foreign_male_counts', 'foreign_female_counts', 'total_counts',
       'include_foreign_household_counts', 'facility_count', 'price_per_sqm',
       'median_price_1year_ago', 'median_price_2year_ago',
       'median_price_3year_ago', 'median_price_4year_ago',
       'median_price_5year_ago', 'expo_announcement_flag',
       'expo_development_start_flag', 'expo_development_completion_flag',
       'expo_implementation_flag', 'ir_announcement_flag',
       'ir_development_sta

In [10]:
# 落とすカラムのリストの作成
drop_list = ['id', 'type', 'price_info_class', 'municipality_code', 'area_no_max_limit_flag', 'time_no_max_limit_flag', 'nearest_station']

# カラム削減クラスの定義
class skPlumberBase(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        # fit済みであるという事実の記録のため
        self.model_ = True
        return self
    def transform(self, X):
        return self
    
class MyProcess(skPlumberBase):
    def __init__(self):
        pass
    def transform(self, df):
        return df.drop(drop_list, axis=1)

# Pipelineのインスタンス化
pipe = Pipeline([("my_process", MyProcess())])

# データを目的変数と特徴量に分ける
y = df2['transaction_price']
X_raw = df2.drop(columns=['transaction_price', 'price_per_sqm'])
X = pipe.fit_transform(X_raw)

print(X.shape, y.shape)


(43804, 37) (43804,)


### 予測と検証

In [11]:
# 手動time_splitの実装

model = ExplainableBoostingRegressor()

folds = [
    (range(2014, 2017), range(2017, 2020)),
    (range(2014, 2020), range(2020, 2023)),
    (range(2014, 2023), range(2023, 2026))
    ]

for i, (train_years, test_years) in enumerate(folds):
    train_mask = df2['transaction_year'].isin(train_years)
    test_mask = df2['transaction_year'].isin(test_years)

    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    score = r2_score(y_test, y_pred)
    print(f'{i+1}回目の検証')
    print(f'MAE: {mae}')
    print(f'MSE: {mse}')
    print(f'MAPE: {mape}')
    print(f'r2_score: {score}')
    


/Users/shoki_u/ml-training/objective/mlit_property_price_prediction/.venv/lib/python3.11/site-packages/interpret/utils/_preprocessor.py:361: RuntimeWarning: All-NaN slice encountered
  min_feature_val = np.nanmin(X_col)
/Users/shoki_u/ml-training/objective/mlit_property_price_prediction/.venv/lib/python3.11/site-packages/interpret/utils/_preprocessor.py:362: RuntimeWarning: All-NaN slice encountered
  max_feature_val = np.nanmax(X_col)
/Users/shoki_u/ml-training/objective/mlit_property_price_prediction/.venv/lib/python3.11/site-packages/interpret/utils/_preprocessor.py:361: RuntimeWarning: All-NaN slice encountered
  min_feature_val = np.nanmin(X_col)
/Users/shoki_u/ml-training/objective/mlit_property_price_prediction/.venv/lib/python3.11/site-packages/interpret/utils/_preprocessor.py:362: RuntimeWarning: All-NaN slice encountered
  max_feature_val = np.nanmax(X_col)
/Users/shoki_u/ml-training/objective/mlit_property_price_prediction/.venv/lib/python3.11/site-packages/interpret/utils/_

1回目の検証
MAE: 4106644.5766319176
MSE: 86627592169519.58
MAPE: 0.24520826490872383
r2_score: 0.6908690958607917


/Users/shoki_u/ml-training/objective/mlit_property_price_prediction/.venv/lib/python3.11/site-packages/interpret/glassbox/_ebm/_ebm.py:871: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(


2回目の検証
MAE: 4417477.850709855
MSE: 81524646348597.69
MAPE: 0.22313899593288797
r2_score: 0.7785024259828011


/Users/shoki_u/ml-training/objective/mlit_property_price_prediction/.venv/lib/python3.11/site-packages/interpret/glassbox/_ebm/_ebm.py:871: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(


3回目の検証
MAE: 6216915.762925735
MSE: 165435568442319.4
MAPE: 0.2382399328102374
r2_score: 0.7077973611818955


In [12]:
# 万博データでのモデルの解釈性の確認
ebm_global = model.explain_global()
show(ebm_global)

<!-- http://127.0.0.1:7001/5111620752/ -->

In [13]:
# area_sqmに立地の情報が含まれている可能性の確認

df2.groupby('municipality')['area_sqm'].mean().sort_values()

municipality
浪速区     30.411738
東淀川区    41.125722
中央区     41.651347
淀川区     43.297178
東成区      43.85664
福島区     45.242588
西区      45.640109
港区      47.118794
北区       47.79444
大正区     49.064171
生野区     49.641791
西淀川区    51.560403
天王寺区    53.741294
西成区     55.436242
東住吉区    55.924138
都島区     57.647332
城東区     60.072431
旭区        60.9375
阿倍野区    61.491792
平野区     61.676301
住吉区     62.974138
此花区     63.423507
住之江区    68.290973
鶴見区     69.192547
Name: area_sqm, dtype: Float64

In [14]:
# ローカルなデータに対する可視化
ebm_local = model.explain_local(X_test[X_test['municipality'] == '此花区'].iloc[[0]],
                                  y_test[X_test['municipality'] == '此花区'].iloc[[0]])
show(ebm_local)

<!-- http://127.0.0.1:7001/4939419792/ -->

### df3の予測

In [15]:
# df3の不要な列を落とし、順番をXと揃える

X_df3 = pipe.transform(df3)
X_df3 = X_df3[X.columns.tolist()]

print(X.columns.tolist())
print(X_df3.columns.tolist())
print(X.columns.equals(X_df3.columns))

['municipality', 'layout', 'structure', 'use', 'future_use', 'city_planning', 'renovation', 'transaction_notes', 'coverage_ratio', 'floor_area_ratio', 'area_sqm', 'building_year', 'building_before_war_flag', 'time_range_write_flag', 'time_to_station', 'transaction_year', 'transaction_quarter', 'actual_foreign_guests', 'foreign_male_counts', 'foreign_female_counts', 'total_counts', 'include_foreign_household_counts', 'facility_count', 'median_price_1year_ago', 'median_price_2year_ago', 'median_price_3year_ago', 'median_price_4year_ago', 'median_price_5year_ago', 'expo_announcement_flag', 'expo_development_start_flag', 'expo_development_completion_flag', 'expo_implementation_flag', 'ir_announcement_flag', 'ir_development_start_flag', 'ir_development_completion_flag', 'ir_implementation_flag', 'distance_to_yumeshima_km']
['municipality', 'layout', 'structure', 'use', 'future_use', 'city_planning', 'renovation', 'transaction_notes', 'coverage_ratio', 'floor_area_ratio', 'area_sqm', 'buildi

In [16]:
# 比較用の2025データの作成

df3_2025 = df3.copy()
df3_2025['transaction_year'] = 2025
df3_2025 = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df3_2025)
df3_2025 = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df3_2025)

In [17]:
years_to_predict = [2025, 2026, 2027, 2028, 2029, 2030]
preds = {}

for year in years_to_predict:
    df_year = df3.copy()
    df_year['transaction_year'] = year
    df_year = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df_year)
    df_year = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df_year)
    X_year = pipe.transform(df_year)[X.columns.tolist()]
    preds[year] = model.predict(X_year)

In [18]:
preds

{2025: array([38499484.35264107, 25483208.81951209, 15828707.10251704, ...,
        13867664.75827198, 27892142.73849655, 35153242.36178795],
       shape=(43804,)),
 2026: array([38499484.35264107, 25483208.81951209, 15828707.10251704, ...,
        13867664.75827198, 27892142.73849655, 35153242.36178795],
       shape=(43804,)),
 2027: array([38499484.35264107, 25483208.81951209, 15828707.10251704, ...,
        13867664.75827198, 27892142.73849655, 35153242.36178795],
       shape=(43804,)),
 2028: array([38499484.35264107, 25483208.81951209, 15828707.10251704, ...,
        13867664.75827198, 27892142.73849655, 35153242.36178795],
       shape=(43804,)),
 2029: array([38499484.35264107, 25483208.81951209, 15828707.10251704, ...,
        13867664.75827198, 27892142.73849655, 35153242.36178795],
       shape=(43804,)),
 2030: array([38499484.35264107, 25483208.81951209, 15828707.10251704, ...,
        13867664.75827198, 27892142.73849655, 35153242.36178795],
       shape=(43804,))}

In [19]:
# 各municipality列が同じかどうかの確認
df_year_dict = {}

for year in years_to_predict:
    df_year = df3.copy()
    df_year['transaction_year'] = year
    df_year = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df_year)
    df_year = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df_year)
    df_year_dict[year] = df_year['municipality']

for year in years_to_predict:
    if year - 1 not in years_to_predict:
        continue
    print(df_year_dict[year - 1].equals(df_year_dict[year]))
    

True
True
True
True
True


In [20]:
df_pred = pd.DataFrame([df_year_dict[2025], preds[2025], preds[2026], preds[2027], preds[2028], preds[2029], preds[2030]])
df_pred.T

,municipality,Unnamed 0,Unnamed 1,Unnamed 2,Unnamed 3,Unnamed 4,Unnamed 5
0,都島区,38499484.352641,38499484.352641,38499484.352641,38499484.352641,38499484.352641,38499484.352641
1,都島区,25483208.819512,25483208.819512,25483208.819512,25483208.819512,25483208.819512,25483208.819512
2,都島区,15828707.102517,15828707.102517,15828707.102517,15828707.102517,15828707.102517,15828707.102517
3,都島区,18188125.344486,18188125.344486,18188125.344486,18188125.344486,18188125.344486,18188125.344486
4,都島区,16508234.836454,16508234.836454,16508234.836454,16508234.836454,16508234.836454,16508234.836454
...,...,...,...,...,...,...,...
43799,中央区,11737032.300933,11737032.300933,11737032.300933,11737032.300933,11737032.300933,11737032.300933
43800,中央区,11505494.173368,11505494.173368,11505494.173368,11505494.173368,11505494.173368,11505494.173368
43801,中央区,13867664.758272,13867664.758272,13867664.758272,13867664.758272,13867664.758272,13867664.758272
43802,中央区,27892142.738497,27892142.738497,27892142.738497,27892142.738497,27892142.738497,27892142.738497


In [21]:
(preds[2029] == preds[2030]).mean()

np.float64(1.0)

In [22]:
# X_yearの確認
df_2029 = df3.copy()
df_2029['transaction_year'] = 2029
df_2029 = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df_2029)
df_2029 = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df_2029)
X_2029 = pipe.transform(df_2029)[X.columns.tolist()]

df_2030 = df3.copy()
df_2030['transaction_year'] = 2030
df_2030 = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df_2030)
df_2030 = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df_2030)
X_2030 = pipe.transform(df_2030)[X.columns.tolist()]

#X_2029['transaction_year'].mean()
#X_2030['transaction_year'].mean()
#X_2029[X_2029['municipality'] == '此花区']['ir_implementation_flag'].mean()
#X_2030[X_2030['municipality'] == '此花区']['ir_implementation_flag'].mean()

In [23]:
idx = X_2029[df3['municipality'] == '此花区'].index[0]  # 或いは適当な行番号

local_2029 = model.explain_local(X_2029.loc[[idx]], y.loc[[idx]])
local_2030 = model.explain_local(X_2030.loc[[idx]], y.loc[[idx]])

In [24]:
show(local_2030)

<!-- http://127.0.0.1:7001/4979269200/ -->

### ラグを逐次生成する形にする

In [25]:
# 2021年から2025年までの区ごとの平米数単位取引価格を取り出す
median_history = {}

df_hist = df2[df2['transaction_year'].between(2020, 2025)]
grouped = df_hist.groupby(['municipality', 'transaction_year'])['price_per_sqm'].median().reset_index()

for year in range(2020, 2026):
    median_history[year] = (
        grouped[grouped['transaction_year'] == year].set_index('municipality')['price_per_sqm']
    )

# 2026~2030年の予測を行う

year_list = [2025, 2026, 2027, 2028, 2029, 2030]
preds = {}

for year in year_list:
    df_year = df3.copy()
    # 取引年にyear_listから取り出した各年を代入
    df_year['transaction_year'] = year
    # 1年前から5年前のラグ特徴量の作成
    for lag in range(1, 6):
        ref_year = year - lag

        df_year[f'median_price_{lag}year_ago'] = df_year['municipality'].map(median_history[ref_year])

    # 万博とIRのイベントフラグの構築
    df_year = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df_year)
    df_year = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df_year)

    # 不要な特徴量の削減と予測
    X_year = pipe.transform(df_year)[X.columns.tolist()]
    pred = model.predict(X_year)

    # ラグ特徴量を作成するため、平米数単位予測(取引価格)を計算し、median_historyに格納
    df_year['price_per_sqm'] = pred / df_year['area_sqm']
    if year == 2025 :
        pass
    else:
        median_history[year] = (
    df_year.groupby('municipality')['price_per_sqm'].median()
    )
    # predsに各年の予測を格納
    preds[year] = pred

print(median_history[2026]['此花区'])
print(median_history[2027]['此花区'])
print(median_history[2028]['此花区'])

417654.4956432384
414041.69557081256
419940.1536724594


In [26]:
# 予測値が一致していないかの確認
(preds[2029] == preds[2030]).mean()

np.float64(0.0)

In [27]:
preds[2025]

array([38499484.35264107, 25483208.81951209, 15828707.10251704, ...,
       13867664.75827198, 27892142.73849655, 35153242.36178795],
      shape=(43804,))

In [28]:
# 予測値をdataframe化
df_tmp = pd.DataFrame([df_year['municipality'], preds[2025], preds[2026], preds[2027], preds[2028], preds[2029], preds[2030]])
df_preds = df_tmp.T
df_preds.columns=['municipality', 'pred_2025', 'pred_2026', 'pred_2027', 'pred_2028', 'pred_2029', 'pred_2030']
pred_cols = [f'pred_{year}' for year in year_list]
df_preds[pred_cols] = df_preds[pred_cols].astype(float)
df_preds

,municipality,pred_2025,pred_2026,pred_2027,pred_2028,pred_2029,pred_2030
0,都島区,3.849948e+07,4.225991e+07,4.083493e+07,3.517954e+07,3.886148e+07,3.656661e+07
1,都島区,2.548321e+07,2.446701e+07,2.475496e+07,2.228892e+07,2.576216e+07,2.529259e+07
2,都島区,1.582871e+07,1.449316e+07,1.521415e+07,1.303883e+07,1.660337e+07,1.591744e+07
3,都島区,1.818813e+07,1.861258e+07,1.880148e+07,1.612992e+07,1.883344e+07,1.973730e+07
4,都島区,1.650823e+07,1.633336e+07,1.624834e+07,1.434540e+07,1.734811e+07,1.734915e+07
...,...,...,...,...,...,...,...
43799,中央区,1.173703e+07,1.173703e+07,1.110095e+07,1.089748e+07,1.206717e+07,1.355219e+07
43800,中央区,1.150549e+07,1.150549e+07,1.097439e+07,1.074483e+07,1.186252e+07,1.325624e+07
43801,中央区,1.386766e+07,1.386766e+07,1.323537e+07,1.268587e+07,1.437482e+07,1.594038e+07
43802,中央区,2.789214e+07,2.789214e+07,2.630682e+07,2.543771e+07,2.512965e+07,2.606566e+07


In [29]:
median_history

{2020: municipality
 中央区          650000.0
 住之江区    213333.333333
 住吉区     338461.538462
 北区           680000.0
 城東区          350000.0
 大正区     390909.090909
 天王寺区    523076.923077
 平野区     266666.666667
 旭区      226086.956522
 東住吉区    333333.333333
 東成区     436666.666667
 東淀川区    309545.454545
 此花区     338461.538462
 浪速区          700000.0
 淀川区          395000.0
 港区           640000.0
 生野区          240000.0
 福島区     576470.588235
 西区           600000.0
 西成区     213333.333333
 西淀川区    303846.153846
 都島区          400000.0
 阿倍野区         440000.0
 鶴見区     353846.153846
 Name: price_per_sqm, dtype: Float64,
 2021: municipality
 中央区     685714.285714
 住之江区    241826.923077
 住吉区     351923.076923
 北区           700000.0
 城東区          400000.0
 大正区          600000.0
 天王寺区    548333.333333
 平野区     266666.666667
 旭区      276960.784314
 東住吉区    353846.153846
 東成区     503333.333333
 東淀川区    326666.666667
 此花区     381818.181818
 浪速区          700000.0
 淀川区     415384.615385
 港区           560000.0
 生

In [30]:
print(median_history[2024]['中央区'])  # 2025年予測で使う1年前
print(median_history[2025]['中央区'])  # 2026年予測で使う1年前

800000.0
850000.0


In [31]:
df_2025 = df3.copy()
df_2025['transaction_year'] = 2025
# 1年前から5年前のラグ特徴量の作成
for lag in range(1, 6):
    ref_year = 2025 - lag

    df_2025[f'median_price_{lag}year_ago'] = df_2025['municipality'].map(median_history[ref_year])

# 万博とIRのイベントフラグの構築
df_2025 = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df_2025)
df_2025 = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df_2025)

# 不要な特徴量の削減と予測
X_2025 = pipe.transform(df_2025)[X.columns.tolist()]
pred_2025 = model.predict(X_2025)

# ラグ特徴量を作成するため、平米数単位予測(取引価格)を計算し、median_historyに格納
df_2025['price_per_sqm'] = pred_2025 / df_2025['area_sqm']



df_2026 = df3.copy()
df_2026['transaction_year'] = 2026
# 1年前から5年前のラグ特徴量の作成
for lag in range(1, 6):
    ref_year = 2026 - lag

    df_2026[f'median_price_{lag}year_ago'] = df_2026['municipality'].map(median_history[ref_year])

# 万博とIRのイベントフラグの構築
df_2026 = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df_2026)
df_2026 = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df_2026)

# 不要な特徴量の削減と予測
X_2026 = pipe.transform(df_2026)[X.columns.tolist()]
pred_2026 = model.predict(X_2026)

# ラグ特徴量を作成するため、平米数単位予測(取引価格)を計算し、median_historyに格納
df_2026['price_per_sqm'] = pred_2026 / df_2026['area_sqm']


idx = X_2025[df3['municipality'] == '中央区'].index[0]  # 或いは適当な行番号

local_2025 = model.explain_local(X_2025.loc[[idx]], y.loc[[idx]])
local_2026 = model.explain_local(X_2026.loc[[idx]], y.loc[[idx]])

In [32]:
show(local_2026)

<!-- http://127.0.0.1:7001/4980904528/ -->

In [52]:
year_list = [2025, 2026, 2027, 2028, 2029, 2030]
median_preds = {}

for year in year_list:
    df_preds[f'rate_of_increase{year}'] = (df_preds[f'pred_{year}'] - df_preds['pred_2025']) / df_preds['pred_2025'] * 100 + 100
    median_preds[year] = (df_preds.groupby('municipality')[f'rate_of_increase{year}'].median())
    

In [34]:
print(f'【2025年から2030年までの区ごとの価格上昇率】')
median_preds[2030].sort_values(ascending=False)

【2025年から2030年までの区ごとの価格上昇率】


municipality
北区      117.567098
中央区     110.513904
東淀川区    103.742368
東住吉区    102.614987
此花区     102.519421
浪速区     102.480241
鶴見区     102.283127
住吉区     101.530261
淀川区      99.633158
城東区      99.538304
住之江区     99.526627
平野区      99.418262
西成区      98.504554
港区       97.878056
都島区      97.798615
東成区      96.626786
福島区      96.597362
阿倍野区     96.045095
旭区       95.527049
生野区      95.180961
西淀川区     94.124590
天王寺区     93.373763
西区       86.759342
大正区      83.012831
Name: rate_of_increase2030, dtype: float64

In [ ]:
# Tableauでの可視化用に、各年の各区の予測値の中央値を縦持ちにする
year_list = [2025, 2026, 2027, 2028, 2029, 2030]
df_list = []

for i, year in enumerate(year_list):
    df_year = median_preds[year].reset_index()
    df_year.columns = ['municipality', 'rate_of_increase']
    df_year['year'] = year
    df_list.append(df_year)

df_median_preds = pd.concat(df_list, axis=0)

In [57]:
# 縦持ちのdfをcsv化
df_median_preds.to_csv('../data/processed/median_preds_by_year.csv', index=False)